# Repository, CLI, And Git-Compatible Workflows

Core examples for signed repository creation, asset pushes, version materialization, CLI execution, and the trusted-host Git compatibility surface.

Run this cell from the repository root after `npm ci` and `npm run build`. The stored output below was regenerated by `npm run notebooks:build`.

## Create, Record, Push, Verify, And Materialize

**Use when:** Use this flow when an application embeds the Core SDK and wants signed history plus a deployable snapshot.

The next cell is the executable example.

In [1]:
import { mkdtempSync, writeFileSync, existsSync, readFileSync } from 'node:fs';
import { tmpdir } from 'node:os';
import { join } from 'node:path';
import { EpochRepository } from 'epoch';

const root = mkdtempSync(join(tmpdir(), 'epoch-notebook-core-'));
writeFileSync(join(root, 'README.md'), '# Demo\n');
writeFileSync(join(root, 'app.js'), 'console.log("hello epoch")\n');

const repository = EpochRepository.openOrCreate(root, { author: 'alice' });
const recorded = repository.recordFile('README.md', 'text/markdown');
const pushed = repository.push(['app.js'], { author: 'alice', version: 'site-v1' });
const materialized = repository.materializeVersion('site-v1', { outDir: join(root, 'deploy') });
const problems = repository.verify();

console.log(JSON.stringify({
  author: repository.identity(),
  firstEventType: recorded.type,
  pushedFiles: pushed.recorded.length,
  versionName: pushed.version?.payload.name,
  materializedFiles: materialized.files,
  deployedAppExists: existsSync(join(root, 'deploy', 'app.js')),
  deployedApp: readFileSync(join(root, 'deploy', 'app.js'), 'utf8').trim(),
  verifyProblems: problems.length,
}, null, 2));

{
  "author": "alice",
  "firstEventType": "record",
  "pushedFiles": 1,
  "versionName": "site-v1",
  "materializedFiles": [
    "README.md",
    "app.js"
  ],
  "deployedAppExists": true,
  "deployedApp": "console.log(\"hello epoch\")",
  "verifyProblems": 0
}


**How to read the output:** The output shows one signed record event, one pushed app file, a named version, materialized files on disk, and a clean verification result.

## Run The Source CLI

**Use when:** Use this flow when validating the command-line contract from a source checkout.

The next cell is the executable example.

In [2]:
import { execFileSync } from 'node:child_process';
import { mkdtempSync, writeFileSync, readFileSync, existsSync } from 'node:fs';
import { tmpdir } from 'node:os';
import { join } from 'node:path';

const root = mkdtempSync(join(tmpdir(), 'epoch-notebook-cli-'));
writeFileSync(join(root, 'notes.txt'), 'release notes\n');
const cli = ['packages/Epoch.CLI/dist/cli.js'];

function epoch(args) {
  return execFileSync(process.execPath, [...cli, '--repo', root, ...args], { encoding: 'utf8' }).trim();
}

const init = epoch(['init', '--author', 'alice']);
epoch(['record', '--type', 'text/plain', 'notes.txt']);
const verify = epoch(['verify']);
const version = epoch(['version', 'create', 'release-notes']);
const outDir = join(root, 'exported');
const materialize = epoch(['version', 'materialize', 'release-notes', '--out', outDir]);
const events = epoch(['events']).split('\n').map((line) => line.split(' ')[1]);

console.log(JSON.stringify({
  init: init.replace(root, '<repo>'),
  eventTypes: events,
  verify,
  version: version.replace(/[a-f0-9]{64}/, '<event-id>'),
  materialize: materialize.replace(outDir, '<out>'),
  exportedNotes: existsSync(join(outDir, 'notes.txt')) ? readFileSync(join(outDir, 'notes.txt'), 'utf8').trim() : 'missing',
}, null, 2));

{
  "init": "initialized Epoch repository at <repo>\\.epoch",
  "eventTypes": [
    "record",
    "operation",
    "version"
  ],
  "verify": "ok",
  "version": "version release-notes <event-id>",
  "materialize": "materialized version release-notes to <out>",
  "exportedNotes": "release notes"
}


**How to read the output:** The CLI output is normalized where event IDs and temporary paths would otherwise vary. It proves init, record, verify, version creation, and materialization work through the built CLI.

## Use Git-Compatible Host Commands

**Use when:** Use this flow when a trusted host filesystem needs Git-like add, commit, status, and explicit unsupported-operation handling.

The next cell is the executable example.

In [3]:
import { execFileSync } from 'node:child_process';
import { mkdtempSync, writeFileSync } from 'node:fs';
import { tmpdir } from 'node:os';
import { join } from 'node:path';
import { EpochCoreGit, unsupported } from 'epoch/Epoch.Core.Git';

const root = mkdtempSync(join(tmpdir(), 'epoch-notebook-git-'));
execFileSync('git', ['-C', root, '-c', 'init.defaultBranch=main', 'init'], { stdio: 'pipe' });
execFileSync('git', ['-C', root, 'config', 'user.name', 'Notebook'], { stdio: 'pipe' });
execFileSync('git', ['-C', root, 'config', 'user.email', 'notebook@example.invalid'], { stdio: 'pipe' });
writeFileSync(join(root, '.gitkeep'), '');
execFileSync('git', ['-C', root, 'add', '.gitkeep'], { stdio: 'pipe' });
execFileSync('git', ['-C', root, 'commit', '-m', 'Initial git history'], { stdio: 'pipe' });

const git = EpochCoreGit.init(root, { author: 'alice', remote: 'https://example.invalid/acme/project.git' });
writeFileSync(join(root, 'README.md'), '# Project\n');
git.add(['README.md']);
const commit = git.commit('Add README', { author: 'alice' });
const status = git.status();
const unsupportedError = unsupported('push');

console.log(JSON.stringify({
  provider: git.remote()?.provider,
  remote: git.remote()?.remote,
  recordedFiles: commit.recorded.length,
  commitEventType: commit.event.type,
  statusMentionsVerify: status.stdout.includes('Epoch repository verifies successfully'),
  unsupportedName: unsupportedError.name,
  unsupportedMessage: unsupportedError.message,
}, null, 2));

{
  "provider": "git",
  "remote": "https://example.invalid/acme/project.git",
  "recordedFiles": 1,
  "commitEventType": "git.commit",
  "statusMentionsVerify": true,
  "unsupportedName": "UnsupportedGitOperationError",
  "unsupportedMessage": "git push is not supported by Epoch Git compatibility: there is no safe Epoch operation or clear workaround for this Git command yet"
}


**How to read the output:** The Git-compatible layer records staged files into Epoch history and reports unsupported commands as explicit typed errors instead of pretending native Git semantics exist everywhere.